<a href="https://colab.research.google.com/github/gcbenito1-blip/elective_streamlit/blob/deploy/data_preprocessing1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install translate
!pip install google-play-scraper
!pip install langdetect
!pip install translate
!pip install fast-langdetect
!pip install clean-text[gpl]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.0 MB/s eta 0:00:00


In [44]:
from google_play_scraper import Sort, reviews_all
from langdetect import detect
# from fast_langdetect import detect_language
import pandas as pd
import unicodedata
import re

In [27]:
def scrape_data():

    results = reviews_all(
        'egov.app',
        lang='en',
        sort=Sort.NEWEST,
    )

    return results

def detect_lang(row):
    try:
        if detect(row['content']) == 'tl':
            return 1
        else:
            return 0
    except:
        return 0

In [28]:
review_dataset = scrape_data()
df = pd.DataFrame(review_dataset)

df = df.reindex(columns=[
    'reviewId', 'content', 'score', 'thumbsUpCount',
    'reviewCreatedVersion', 'at'
])

df.index.name = 'index'
df['content'] = df['content'].astype(str).str.lower()
df['is_tagalog'] = df.apply(detect_lang, axis=1)
df.to_csv('all_reviews.csv')

In [45]:
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,is_tagalog,clean
index,,,,,,,,
0,8b1f3376-704d-404b-a6df-31669ec01bc4,thanks this apps. because for me this apps is ...,5,0,2.7.1,2026-04-16 11:20:49,0,thanks this apps. because for me this apps is ...
1,96d73a45-1777-49ef-8928-118a48d7b367,"happy,easy and fast ,and even first time to cr...",5,0,2.7.1,2026-04-16 10:23:03,0,"happy,easy and fast ,and even first time to cr..."
2,3728b718-bafd-4638-aa74-53fec7d7558d,perfect,5,0,2.7.1,2026-04-16 09:13:39,0,perfect
3,2f82efd1-848f-4b33-b10e-69f4479ca728,absolutely great...,5,0,None,2026-04-16 09:06:02,0,absolutely great...
4,9b9c244d-df83-4915-a7c2-18fbfb637904,ok mas mapamadali ang gusto natin thru egov...,4,0,2.7.1,2026-04-16 08:52:50,1,ok mas mapamadali ang gusto natin thru egov...
...,...,...,...,...,...,...,...,...
44511,d039976d-d0d8-4867-8a57-9b7b568ce65f,nice! first app ng gov!,5,1,1.1.2,2023-06-02 02:38:43,0,nice! first app ng gov!
44512,1abdfb9f-eb73-45b8-a6ea-e8ab42b1b0c3,first!,5,0,1.1.2,2023-06-02 02:13:54,0,first!
44513,8fb3ca76-7748-4a25-a171-a3f112bac57d,please put in suffix the ii after sr jr it fol...,1,20,1.1.1,2023-06-01 12:04:40,0,please put in suffix the ii after sr jr it fol...


In [36]:
# from cleantext import clean
# df['clean'] = df['content'].apply(
#     lambda x: clean(
#         x,
#         fix_unicode=True,
#         no_line_breaks=True,
#         no_code=True,
#         no_urls=True,
#         no_emails=True,
#         no_phone_numbers=True,
#         no_ip_addresses=True,
#         no_file_paths=True,
#         no_emoji=True,
#         no_currency_symbols=True
#     )
# )

In [46]:
def clean_text(text):
    text = unicodedata.normalize('NFKC', str(text))#unicode formatting
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)#http
    text = re.sub(r'@\w+|#\w+', '', text) #removed mentions/hashtags
    text = re.sub(r'<.*?>', '', text) #removed html
    text = re.sub(r'[^\w\s\.!?@#]', '', text) #symbols
    text = re.sub(r'\s+', ' ', text) #space normalization
    return text.lower().strip() #lowerspace and whitespace trimming

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE
)

def remove_emoji(text):
    return emoji_pattern.sub('', str(text))

df = df.copy()

df['clean'] = (
    df['clean']
    .astype(str)
    .apply(remove_emoji)
    .apply(clean_text)
)

# Remove empty rows AFTER cleaning
df = df[df['clean'].notna() & (df['clean'].str.strip() != '')]

df.to_csv('clean_dataset.csv')

In [49]:
df1 = pd.read_csv('clean_dataset.csv')
df_eng = df1[df1['is_tagalog'] ==0].copy()
df_eng['translated'] = df_eng['clean']
df_eng.to_csv('eng_review.csv')
df_tgl = df1[df1['is_tagalog'] ==1].copy()
df_tgl.to_csv('tgl_review.csv')

In [50]:
df_tgl

,index,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,is_tagalog,clean
4,4,9b9c244d-df83-4915-a7c2-18fbfb637904,ok mas mapamadali ang gusto natin thru egov...,4,0,2.7.1,2026-04-16 08:52:50,1,ok mas mapamadali ang gusto natin thru egov...
11,11,7d529fed-fb53-46be-96bc-75d677a6e07f,maganda app,5,0,2.7.1,2026-04-16 05:22:59,1,maganda app
14,14,8263a442-56cb-4d10-a93c-a26636278afe,"ambagal, please improve it. anlaki ng budget n...",1,0,2.7.1,2026-04-16 04:17:16,1,ambagal please improve it. anlaki ng budget ny...
27,28,06a0beb8-eeed-46b7-8248-557ac320d026,akala ko hindi ko ma verify ang aking gcash da...,5,0,2.7.1,2026-04-16 02:10:47,1,akala ko hindi ko ma verify ang aking gcash da...
28,29,a3ac8d5c-d458-438c-9505-1c3f29636c7b,ang ganda nakikita ko ulit ang national i'd ko,5,0,2.7.1,2026-04-16 02:05:59,1,ang ganda nakikita ko ulit ang national id ko
...,...,...,...,...,...,...,...,...,...
43872,44496,7c4c8f86-a802-4e7f-8fd8-8582a75e0aba,"di ako maka veryfy, ayaw mag scan ng philsys k...",2,1,1.1.2,2023-06-02 09:25:59,1,di ako maka veryfy ayaw mag scan ng philsys ko...
43879,44503,3f2059c6-71c6-4f2e-bdc7-451621c730de,it's amazing 😍😍😍,5,1,1.1.2,2023-06-02 07:01:57,1,its amazing
43880,44504,eb3c7898-d67f-4318-a2c8-405d6127a2db,"accessible at madaling gamitin, ganda gamitin ...",5,1,1.1.2,2023-06-02 06:47:04,1,accessible at madaling gamitin ganda gamitin d...
43885,44509,ffe92e5d-c018-4502-a834-c7d48eecac53,"ayaw magproceed after magcaptuee ng id, may bo...",3,5,1.1.2,2023-06-02 03:45:24,1,ayaw magproceed after magcaptuee ng id may box...
